LSTM + Attention Mechanism for clustering latent dim

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import os
import tensorflow as tf
from tensorflow.keras import Model, Input, regularizers
from tensorflow.keras.layers import (
    LSTM, Dense, Flatten, Activation, RepeatVector, Permute, Multiply, 
    Lambda, Dropout, BatchNormalization, TimeDistributed
)
from tensorflow.keras.optimizers import Adam

from tensorflow.keras.layers import Layer

class ReduceSumLayer(Layer):
    def call(self, inputs):
        return tf.reduce_sum(inputs, axis=1)

class LSTMAttentionAutoencoder:
    def __init__(self, 
                 input_shape=(51, 2),
                 latent_dim=16, 
                 lstm_units=[64, 32],
                 dropout_rate=0.3, 
                 l2_reg=1e-4):
        
        self.input_shape = input_shape
        self.latent_dim = latent_dim
        self.lstm_units = lstm_units
        self.dropout_rate = dropout_rate
        self.l2_reg = l2_reg
        self.model, self.encoder = self._build_model()
    
    def _attention_block(self, lstm_out):
        attention = Dense(1, activation='tanh')(lstm_out)
        attention = Flatten()(attention)
        attention = Activation('softmax')(attention)
        attention = RepeatVector(self.lstm_units[-1])(attention)
        attention = Permute([2, 1])(attention)
        weighted = Multiply()([lstm_out, attention])
        context_vector = ReduceSumLayer()(weighted)
        return context_vector

    def _build_model(self):
        # Encoder
        inputs = Input(shape=self.input_shape)
        x = inputs
        for i, units in enumerate(self.lstm_units):
            x = LSTM(units, return_sequences=True,
                     kernel_regularizer=regularizers.l2(self.l2_reg))(x)
            x = BatchNormalization()(x)
            x = Dropout(self.dropout_rate)(x)
        context_vector = self._attention_block(x)
        x = Dense(64, activation='relu', kernel_regularizer=regularizers.l2(self.l2_reg))(context_vector)
        x = BatchNormalization()(x)
        x = Dropout(self.dropout_rate)(x)
        latent = Dense(self.latent_dim, name='latent_vector')(x)

        # Decoder
        x = Dense(64, activation='relu')(latent)
        x = RepeatVector(self.input_shape[0])(x)  # Repeat for time steps
        for units in reversed(self.lstm_units):
            x = LSTM(units, return_sequences=True)(x)
            x = BatchNormalization()(x)
            x = Dropout(self.dropout_rate)(x)
        outputs = TimeDistributed(Dense(self.input_shape[1]))(x)  # Output shape: (batch, 51, 2)

        # Models
        autoencoder = Model(inputs, outputs, name="LSTM_Attention_Autoencoder")
        encoder = Model(inputs, latent, name="Encoder")
        return autoencoder, encoder

    def compile(self, learning_rate=1e-3):
        self.model.compile(optimizer=Adam(learning_rate), loss='mse')

    def train(self, X_train, epochs=50, batch_size=16, verbose=1):
        self.model.fit(X_train, X_train, epochs=epochs, batch_size=batch_size, verbose=verbose)

    def get_latent_vectors(self, X):
        return self.encoder.predict(X)

    def summary(self):
        self.model.summary()
#loading data 
data = pd.read_csv(r"C:\Users\Admin\Desktop\CP\Data\processed\final_processed\GAIT_analysis_dataset.csv")
data
#split into train and test data
# Get unique patient IDs
unique_patients = data['Patient ID'].unique()

# Split patient IDs into train and test sets (e.g., 80% train, 20% test)
train_patients, test_patients = train_test_split(unique_patients, test_size=0.25, random_state=7)

# Create train and test DataFrames
train_df = data[data['Patient ID'].isin(train_patients)].reset_index(drop=True)
test_df = data[data['Patient ID'].isin(test_patients)].reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train patients:", train_df['Patient ID'].nunique())
print("Test patients:", test_df['Patient ID'].nunique())

# Before extracting patient_sequences, fill NaNs in train_df and test_df
train_df = train_df.interpolate(axis=0).fillna(method='bfill').fillna(method='ffill')
test_df = test_df.interpolate(axis=0).fillna(method='bfill').fillna(method='ffill')
hip_cols = [col for col in train_df.columns if col.startswith('aSagH_')]
knee_cols = [col for col in train_df.columns if col.startswith('aSagK_')]

patient_sequences = []
patient_ids = []

for pid in train_df['Patient ID'].unique():
    hip_data = train_df[train_df['Patient ID'] == pid][hip_cols].values.flatten()
    knee_data = train_df[train_df['Patient ID'] == pid][knee_cols].values.flatten()
    # Stack as (51, 2): columns are [knee, hip]
    if len(hip_data) == 51 and len(knee_data) == 51:
        patient_sequence = np.stack([knee_data, hip_data], axis=1)  # shape (51, 2)
        patient_sequences.append(patient_sequence)
        patient_ids.append(pid)

# Convert to numpy array
X_train = np.array(patient_sequences)
X_train_patient_ids = np.array(patient_ids)
print("Patient sequences shape:", X_train.shape)
print("Patient IDs shape:", X_train_patient_ids.shape)
hip_cols = [col for col in test_df.columns if col.startswith('aSagH_')]
knee_cols = [col for col in test_df.columns if col.startswith('aSagK_')]

patient_sequences = []
patient_ids = []

for pid in test_df['Patient ID'].unique():
    hip_data = test_df[test_df['Patient ID'] == pid][hip_cols].values.flatten()
    knee_data = test_df[test_df['Patient ID'] == pid][knee_cols].values.flatten()
    # Stack as (51, 2): columns are [knee, hip]
    if len(hip_data) == 51 and len(knee_data) == 51:
        patient_sequence = np.stack([knee_data, hip_data], axis=1)  # shape (51, 2)
        patient_sequences.append(patient_sequence)
        patient_ids.append(pid)

# Convert to numpy array
X_test = np.array(patient_sequences)
X_test_patient_ids = np.array(patient_ids)
print("Patient sequences shape:", X_test.shape)
print("Patient IDs shape:", X_test_patient_ids.shape)
#scaling the data

# Reshape to 2D for scaling: (num_patients * 51, 2)
X_train_reshaped = X_train.reshape(-1, X_train.shape[-1])
X_test_reshaped = X_test.reshape(-1, X_test.shape[-1])

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_reshaped)
X_test_scaled = scaler.transform(X_test_reshaped)

# Reshape back to original shape
X_train = X_train_scaled.reshape(X_train.shape)
X_test = X_test_scaled.reshape(X_test.shape)
#scaling the data

# Reshape to 2D for scaling: (num_patients * 51, 2)
X_train_reshaped = X_train.reshape(-1, X_train.shape[-1])
X_test_reshaped = X_test.reshape(-1, X_test.shape[-1])

scaler = MinMaxScaler()
import keras_tuner as kt

def build_model(hp):
    latent_dim = hp.Int('latent_dim', min_value=8, max_value=64, step=8)
    lstm_units_1 = hp.Int('lstm_units_1', min_value=16, max_value=128, step=16)
    lstm_units_2 = hp.Int('lstm_units_2', min_value=8, max_value=64, step=8)
    dropout_rate = hp.Float('dropout_rate', min_value=0.1, max_value=0.5, step=0.1)
    l2_reg = hp.Float('l2_reg', min_value=1e-5, max_value=1e-3, sampling='log')
    
    model = LSTMAttentionAutoencoder(
        input_shape=(51, 2),
        latent_dim=latent_dim,
        lstm_units=[lstm_units_1, lstm_units_2],
        dropout_rate=dropout_rate,
        l2_reg=l2_reg
    )
    model.compile()
    return model.model  # Return the autoencoder (not just encoder)
X_train_scaled = scaler.fit_transform(X_train_reshaped)
X_test_scaled = scaler.transform(X_test_res
tuner = kt.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=10,
    executions_per_trial=1,
    directory=r'C:\Users\Admin\Desktop\CP\Outputs\HP Tuning\tuner_results',
    project_name='lstm_attention_autoencoder'
)
tuner = kt.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=10,
    executions_per_trial=1,
tuner.search(X_train, X_train, epochs=30, batch_size=16, validation_split=0.2)
tuner.search(X_train, X_train, epochs=30, batch_size=16, validation_split=0.2)
    directory=r'C:\Users\Admin\Desktop\CP\Outputs\HP Tuning\tuner_results',
    project_name='lstm_attention_autoencoder'
)haped)

# Reshape back to original shape
X_train = X_train_scaled.reshape(X_train.shape)
X_test = X_test_scaled.reshape(X_test.shape)
hip_cols = [col for col in test_df.columns if col.startswith('aSagH_')]
knee_cols = [col for col in test_df.columns if col.startswith('aSagK_')]

patient_sequences = []
patient_ids = []

for pid in test_df['Patient ID'].unique():
    hip_data = test_df[test_df['Patient ID'] == pid][hip_cols].values.flatten()
    knee_data = test_df[test_df['Patient ID'] == pid][knee_cols].values.flatten()
    # Stack as (51, 2): columns are [knee, hip]
    if len(hip_data) == 51 and len(knee_data) == 51:
        patient_sequence = np.stack([knee_data, hip_data], axis=1)  # shape (51, 2)
        patient_sequences.append(patient_sequence)
        patient_ids.append(pid)

# Convert to numpy array
X_test = np.array(patient_sequences)
X_test_patient_ids = np.array(patient_ids)
print("Patient sequences shape:", X_test.shape)
print("Patient IDs shape:", X_test_patient_ids.shape)
hip_cols = [col for col in train_df.columns if col.startswith('aSagH_')]
knee_cols = [col for col in train_df.columns if col.st
best_model.save(r'C:\Users\Admin\Desktop\CP\Outputs\HP Tuning\model\best_lstm_attention_autoencoder.h5')
best_model.save(r'C:\Users\Admin\Desktop\CP\Outputs\HP Tuning\model\best_lstm_attention_autoencoder.keras')artswith('aSagK_')]

patient_sequences = []
patient_ids = []

for pid in train_df['Patient ID'].unique():
    hip_data = train_df[train_df['Patient ID'] == pid][hip_cols].values.flatten()
    knee_data = train_df[train_df['Patient ID'] == pid][knee_cols].values.flatten()
    # Stack as (51, 2): columns are [knee, hip]
    if len(hip_data) == 51 and len(knee_data) == 51:
        patient_sequence = np.stack([knee_data, hip_data], axis=1)  # shape (51, 2)
        patient_sequences.append(patient_sequence)
        patient_ids.append(pid)

# Convert to numpy array
X_train = np.array(patient_sequences)
X_train_patient_ids = np.array(patient_ids)
print("Patient sequences shape:", X_train.shape)
print("Patient IDs shape:", X_train_patient_ids.shape)
import matplotlib.pyplot as plt
import numpy as np

X_test_pred = best_model.model.predict(X_test)
idx = 0  # test patient index

actual = X_test[idx]
reconstructed = X_test_pred[idx]

plt.figure(figsize=(10, 5))
plt.plot(actual[:, 0], label='Actual Knee')
plt.plot(reconstructed[:, 0], label='Reconstructed Knee', linestyle='--')
plt.plot(actual[:, 1], label='Actual Hip')
plt.plot(reconstructed[:, 1], label='Reconstructed Hip', linestyle='--')
plt.title('Actual vs Reconstructed Knee and Hip Angles')
plt.xlabel('Frame')
plt.ylabel('Angle (scaled)')
plt.legend()
plt.tight_layout()
plt.savefig(r'C:\Users\Admin\Desktop\CP\Outputs\results\plots\actual_vs_reconstructed_single.png')
plt.show()

from sklearn.metrics import mean_squared_error, mean_absolute_error

# Compute MSE and MAE for each test sample
mse_list = [mean_squared_error(X_test[i], X_test_pred[i]) for i in range(len(X_test))]
mae_list = [mean_absolute_error(X_test[i], X_test_pred[i]) for i in range(len(X_test))]

plt.figure(figsize=(10, 4))
plt.plot(mse_list, label='Test MSE per sample')
plt.plot(mae_list, label='Test MAE per sample')
plt.xlabel('Test Sample Index')
plt.ylabel('Error')
plt.title('Reconstruction Error on Test Set')
plt.legend()
plt.tight_layout()
plt.savefig(r'C:\Users\Admin\Desktop\CP\Outputs\results\plots\test_reconstruction_error.png')
plt.show()


plt.figure(figsize=(10, 4))
plt.hist(mse_list, bins=20, alpha=0.7, label='MSE')
plt.hist(mae_list, bins=20, alpha=0.7, label='MAE')
plt.xlabel('Error')
plt.ylabel('Frequency')
plt.title('Distribution of Reconstruction Errors (Test Set)')
plt.legend()
plt.tight_layout()
plt.savefig(r'C:\Users\Admin\Desktop\CP\Outputs\results\plots\test_error_histogram.png')
plt.show()


plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.axhline(np.mean(mse_list), color='red', linestyle='--', label='Test MSE (mean)')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Training, Validation, and Test Loss')
plt.legend()
plt.tight_layout()
plt.savefig(r'C:\Users\Admin\Desktop\CP\Outputs\results\plots\loss_curve_with_test.png')
plt.show()
# Get latent vectors for train and test sets
X_train_latent = best_model.encoder.predict(X_train)
X_test_latent = best_model.encoder.predict(X_test)

print("Train latent shape:", X_train_latent.shape)
print("Test latent shape:", X_test_latent.shape)
from sklearn.cluster import KMeans

# Choose number of clusters (e.g., 2 or 3)
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
clusters = kmeans.fit_predict(X_train_latent)

print("Cluster assignments for train set:", clusters)
import pandas as pd

# X_train_patient_ids contains the patient IDs in the same order as X_train_latent
cluster_df = pd.DataFrame({
    'Patient ID': X_train_patient_ids,
    'Cluster': clusters
})

print(cluster_df.head())
# Optionally, save to CSV
cluster_df.to_csv(r'C:\Users\Admin\Desktop\CP\Outputs\results\train_patient_clusters.csv', index=False)
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2)
latent_2d = pca.fit_transform(X_train_latent)

plt.figure(figsize=(8,6))
plt.scatter(latent_2d[:,0], latent_2d[:,1], c=clusters, cmap='viridis', s=40)
plt.title('Clustering of Latent Representations (Train Set)')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.colorbar(label='Cluster')
plt.tight_layout()
plt.savefig(r'C:\Users\Admin\Desktop\CP\Outputs\results\plots\latent_clustering.png')
plt.show()
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
latent_tsne = tsne.fit_transform(X_train_latent)

plt.figure(figsize=(8,6))
plt.scatter(latent_tsne[:,0], latent_tsne[:,1], c=clusters, cmap='viridis', s=40)
plt.title('t-SNE of Latent Representations (Train Set)')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.colorbar(label='Cluster')
plt.tight_layout()
plt.savefig(r'C:\Users\Admin\Desktop\CP\Outputs\results\plots\latent_clustering_tsne.png')
plt.show()
import umap

umap_reducer = umap.UMAP(n_components=2, random_state=42)
latent_umap = umap_reducer.fit_transform(X_train_latent)

plt.figure(figsize=(8,6))
plt.scatter(latent_umap[:,0], latent_umap[:,1], c=clusters, cmap='viridis', s=40)
plt.title('UMAP of Latent Representations (Train Set)')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.colorbar(label='Cluster')
plt.tight_layout()
plt.savefig(r'C:\Users\Admin\Desktop\CP\Outputs\results\plots\latent_clustering_umap.png')
plt.show()
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load cluster assignments
cluster_df = pd.read_csv(r'C:\Users\Admin\Desktop\CP\Outputs\results\train_patient_clusters.csv')

# Load original data
data = pd.read_csv(r"C:\Users\Admin\Desktop\CP\Data\processed\final_processed\GAIT_analysis_dataset.csv")

# Merge cluster info into original data
merged = pd.merge(data, cluster_df, on='Patient ID', how='inner')

# Get knee and hip columns
knee_cols = [col for col in merged.columns if col.startswith('aSagK_')]
hip_cols = [col for col in merged.columns if col.startswith('aSagH_')]

# Plot average gait cycle for each cluster
for cluster_id in sorted(merged['Cluster'].unique()):
    cluster_data = merged[merged['Cluster'] == cluster_id]
    # Compute mean across all patients in this cluster for each frame
    knee_mean = cluster_data[knee_cols].groupby(cluster_data['Patient ID']).mean().mean(axis=0)
    hip_mean = cluster_data[hip_cols].groupby(cluster_data['Patient ID']).mean().mean(axis=0)
    
    plt.figure(figsize=(8,4))
    plt.plot(range(1, len(knee_mean)+1), knee_mean, label='Knee')
    plt.plot(range(1, len(hip_mean)+1), hip_mean, label='Hip')
    plt.title(f'Average Gait Cycle for Cluster {cluster_id}')
    plt.xlabel('Frame')
    plt.ylabel('Angle')
    plt.legend()
    plt.tight_layout()
    plt.savefig(rf'C:\Users\Admin\Desktop\CP\Outputs\results\plots\avg_gait_cycle_cluster_{cluster_id}.png')
    plt.show()
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load cluster assignments
cluster_df = pd.read_csv(r'C:\Users\Admin\Desktop\CP\Outputs\results\train_patient_clusters.csv')
data = pd.read_csv(r"C:\Users\Admin\Desktop\CP\Data\processed\final_processed\GAIT_analysis_dataset.csv")
merged = pd.merge(data, cluster_df, on='Patient ID', how='inner')

knee_cols = [col for col in merged.columns if col.startswith('aSagK_')]
hip_cols = [col for col in merged.columns if col.startswith('aSagH_')]

# Prepare average cycles for each cluster
cluster_ids = sorted(merged['Cluster'].unique())
knee_means = []
hip_means = []

for cluster_id in cluster_ids:
    cluster_data = merged[merged['Cluster'] == cluster_id]
    knee_mean = cluster_data[knee_cols].groupby(cluster_data['Patient ID']).mean().mean(axis=0)
    hip_mean = cluster_data[hip_cols].groupby(cluster_data['Patient ID']).mean().mean(axis=0)
    knee_means.append(knee_mean.values)
    hip_means.append(hip_mean.values)

# Plot all clusters' average knee cycles
plt.figure(figsize=(10, 5))
for i, cluster_id in enumerate(cluster_ids):
    plt.plot(range(1, len(knee_means[i])+1), knee_means[i], label=f'Cluster {cluster_id}')
plt.title('Average Knee Gait Cycle by Cluster')
plt.xlabel('Frame')
plt.ylabel('Knee Angle')
plt.legend()
plt.tight_layout()
plt.savefig(r'C:\Users\Admin\Desktop\CP\Outputs\results\plots\avg_gait_cycle_knee_all_clusters.png')
plt.show()

# Plot all clusters' average hip cycles
plt.figure(figsize=(10, 5))
for i, cluster_id in enumerate(cluster_ids):
    plt.plot(range(1, len(hip_means[i])+1), hip_means[i], label=f'Cluster {cluster_id}')
plt.title('Average Hip Gait Cycle by Cluster')
plt.xlabel('Frame')
plt.ylabel('Hip Angle')
plt.legend()
plt.tight_layout()
plt.savefig(r'C:\Users\Admin\Desktop\CP\Outputs\results\plots\avg_gait_cycle_hip_all_clusters.png')
plt.show()
def analyze_gait_dataset(dataset_path, cluster_output_prefix):
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import MinMaxScaler
    from sklearn.cluster import KMeans

    # 1. Load data
    data = pd.read_csv(dataset_path)

    # 2. Split into train/test by patient
    unique_patients = data['Patient ID'].unique()
    train_patients, test_patients = train_test_split(unique_patients, test_size=0.25, random_state=7)
    train_df = data[data['Patient ID'].isin(train_patients)].reset_index(drop=True)
    test_df = data[data['Patient ID'].isin(test_patients)].reset_index(drop=True)

    # 3. Fill NaNs
    train_df = train_df.interpolate(axis=0).fillna(method='bfill').fillna(method='ffill')
    test_df = test_df.interpolate(axis=0).fillna(method='bfill').fillna(method='ffill')

    # 4. Extract sequences
    hip_cols = [col for col in train_df.columns if col.startswith('aSagH_')]
    knee_cols = [col for col in train_df.columns if col.startswith('aSagK_')]

    def get_sequences(df):
        patient_sequences = []
        patient_ids = []
        for pid in df['Patient ID'].unique():
            hip_data = df[df['Patient ID'] == pid][hip_cols].values.flatten()
            knee_data = df[df['Patient ID'] == pid][knee_cols].values.flatten()
            if len(hip_data) == 51 and len(knee_data) == 51:
                patient_sequence = np.stack([knee_data, hip_data], axis=1)
                patient_sequences.append(patient_sequence)
                patient_ids.append(pid)
        return np.array(patient_sequences), np.array(patient_ids)

    X_train, X_train_patient_ids = get_sequences(train_df)
    X_test, X_test_patient_ids = get_sequences(test_df)

    # 5. Scale
    scaler = MinMaxScaler()
    X_train_reshaped = X_train.reshape(-1, X_train.shape[-1])
    X_test_reshaped = X_test.reshape(-1, X_test.shape[-1])
    X_train_scaled = scaler.fit_transform(X_train_reshaped).reshape(X_train.shape)
    X_test_scaled = scaler.transform(X_test_reshaped).reshape(X_test.shape)

    # 6. Train model (reuse your LSTMAttentionAutoencoder class)
    model = LSTMAttentionAutoencoder(
        input_shape=(51, 2),
        latent_dim=24,
        lstm_units=[96, 8],
        dropout_rate=0.1,
        l2_reg=2.3761964881131694e-05
    )
    model.compile()
    model.model.fit(X_train_scaled, X_train_scaled, epochs=30, batch_size=16, validation_split=0.2, verbose=0)

    # 7. Get latent vectors and cluster
    X_train_latent = model.encoder.predict(X_train_scaled)
    n_clusters = 4
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    clusters = kmeans.fit_predict(X_train_latent)

    # 8. Save cluster assignments
    cluster_df = pd.DataFrame({'Patient ID': X_train_patient_ids, 'Cluster': clusters})
    cluster_df.to_csv(f'{cluster_output_prefix}_clusters.csv', index=False)

    # 9. Plot average gait cycles for each cluster (knee and hip, all clusters in one plot each)
    merged = pd.merge(data, cluster_df, on='Patient ID', how='inner')
    knee_cols = [col for col in merged.columns if col.startswith('aSagK_')]
    hip_cols = [col for col in merged.columns if col.startswith('aSagH_')]
    cluster_ids = sorted(merged['Cluster'].unique())
    knee_means = []
    hip_means = []
    for cluster_id in cluster_ids:
        cluster_data = merged[merged['Cluster'] == cluster_id]
        
        knee_mean = cluster_data[knee_cols].groupby(cluster_data['Patient ID']).mean().mean(axis=0)
        hip_mean = cluster_data[hip_cols].groupby(cluster_data['Patient ID']).mean().mean(axis=0)
        knee_means.append(knee_mean.values)
        hip_means.append(hip_mean.values)

    plt.figure(figsize=(10, 5))
    for i, cluster_id in enumerate(cluster_ids):
        plt.plot(range(1, len(knee_means[i])+1), knee_means[i], label=f'Cluster {cluster_id}')
    plt.title('Average Knee Gait Cycle by Cluster')
    plt.xlabel('Frame')
    plt.ylabel('Knee Angle')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'{cluster_output_prefix}_avg_knee.png')
    plt.show()

    plt.figure(figsize=(10, 5))
    for i, cluster_id in enumerate(cluster_ids):
        plt.plot(range(1, len(hip_means[i])+1), hip_means[i], label=f'Cluster {cluster_id}')
    plt.title('Average Hip Gait Cycle by Cluster')
    plt.xlabel('Frame')
    plt.ylabel('Hip Angle')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'{cluster_output_prefix}_avg_hip.png')
    plt.show()



# Example usage:
analyze_gait_dataset(
    r"C:\Users\Admin\Desktop\CP\Data\processed\gait_data_processing\merged_df.csv",
    r"C:\Users\Admin\Desktop\CP\Outputs\results\datasetCPid"
)
# Example usage:
analyze_gait_dataset(
    r"C:\Users\Admin\Desktop\CP\Data\processed\gait_data_processing\closest_side_to_reference.csv",
    r"C:\Users\Admin\Desktop\CP\Outputs\results\datasetCPid_number"
)
import pandas as pd

df_merged = pd.read_csv(r'C:\Users\Admin\Desktop\CP\Outputs\results\train_patient_clusters.csv')
df_original = pd.read_csv(r'C:\Users\Admin\Desktop\CP\Outputs\results\datasetCPid_number_clusters.csv')

# Convert both 'Patient ID' columns to string for safe merging
df_merged['Patient ID'] = df_merged['Patient ID'].astype(str)
df_original['Patient ID'] = df_original['Patient ID'].astype(str)

comparison = pd.merge(df_merged, df_original, on='Patient ID', suffixes=('_merged', '_original'))

diff = comparison[comparison['Cluster_merged'] != comparison['Cluster_original']]
print("Patients with different cluster assignments:")
print(diff)

diff.to_csv(r'C:\Users\Admin\Desktop\CP\Outputs\results\cluster_assignment_differences.csv', index=False)
import pandas as pd

df = pd.read_csv(r"C:\Users\Admin\Desktop\CP\Outputs\results\train_patient_clusters.csv")
df
df['Cluster'].value_counts()
